# PariShiksha: NCERT Science QA Retrieval System

This notebook demonstrates the full RAG pipeline end-to-end:
1. **Stage 1** — Corpus Extraction & Chunking
2. **Stage 2** — Retrieval (BM25 + FAISS)
3. **Stage 3** — Grounded Generation
4. **Stage 4** — Evaluation

## Setup

In [1]:
import os
import sys
import json

# Add src to path
sys.path.insert(0, os.path.join(os.getcwd(), 'src'))

from dotenv import load_dotenv
load_dotenv()

from vec_retrieval import VectorDatabase
print('Setup complete.')

Setup complete.


---
## Corpus Extraction & Chunking

We extracted 12 NCERT Science chapters using `pymupdf4llm` and classified content into 4 types: concept, example, exercise, and solution.

In [12]:
# Load a sample chapter
def load_sample():
    """Function to load a sample chapter"""
    sample_file = 'extracted/iesc110.txt'
    with open(sample_file, 'r') as f:
        sample_text = f.read()

    print(f'Loaded {sample_file}: {len(sample_text)} characters')
    print('\n--- First 500 characters ---')
    print(sample_text[:500])
load_sample()

Loaded extracted/iesc110.txt: 36420 characters

--- First 500 characters ---
## C hapter 

## **10** 

**==> picture [86 x 85] intentionally omitted <==**

## **WORK AND ENERGY** 

In the previous few chapters we have talked about ways of describing the motion of objects, the cause of motion and gravitation. Another concept that helps us understand and interpret many natural phenomena is ‘work’. Closely related to work are energy and power. In this chapter we shall study these concepts. 

All living beings need food.  Living beings have to perform several basic activitie


In [24]:
# Tokenizer Comparison (GPT-2 vs BERT)
from transformers import AutoTokenizer

gpt2 = AutoTokenizer.from_pretrained('gpt2')
bert = AutoTokenizer.from_pretrained('bert-base-uncased')

passages = [
    'The rate of change of velocity is called acceleration.',
    'Every object in the universe attracts every other object with a force.',
    'The cell is the fundamental unit of life.',
    'Matter is made up of particles.',
    'An object moving along a straight line with uniform velocity has zero acceleration.'
]

print(f'{"Passage":<70} | {"GPT-2":>6} | {"BERT":>6}')
print('-' * 90)
for p in passages:
    g_tokens = gpt2.tokenize(p)
    b_tokens = bert.tokenize(p)
    print(f'{p:<70} | {len(g_tokens):>6} | {len(b_tokens):>6}')

Passage                                                                |  GPT-2 |   BERT
------------------------------------------------------------------------------------------
The rate of change of velocity is called acceleration.                 |     10 |     10
Every object in the universe attracts every other object with a force. |     13 |     13
The cell is the fundamental unit of life.                              |      9 |      9
Matter is made up of particles.                                        |      8 |      7
An object moving along a straight line with uniform velocity has zero acceleration. |     14 |     14


In [14]:
# Chunking demonstration
db = VectorDatabase(use_embeddings=False)
chunks = db.chunk_text_bert(sample_text, max_tokens=400, overlap=50)

print(f'Total chunks created: {len(chunks)}')
print(f'Average chunk length: {sum(len(c) for c in chunks) / len(chunks):.0f} chars')
print('\n--- Sample Chunk (first) ---')
print(chunks[0][:300])

Total chunks created: 33
Average chunk length: 1403 chars

--- Sample Chunk (first) ---
# # c hapter * * 7 * * * * = = > picture [ 85 x 86 ] intentionally omitted < = = * * # # * * motion * * in everyday life, we see some objects at rest and others in motion. birds fly, fish swim, blood flows through veins and arteries, and cars move. atoms, molecules, planets, stars and galaxies are a


---
##  Retrieval

We build a chunk store with metadata and implement BM25 retrieval.

In [15]:
# Build the chunk store from all chapters
db = VectorDatabase(use_embeddings=False)
db_path = 'data/vector_db'

if os.path.exists(db_path):
    print('Loading existing database from disk...')
    db.load_from_disk(db_path)
else:
    print('Building database from extracted files...')
    extracted_dir = 'extracted'
    for root, dirs, files in os.walk(extracted_dir):
        if root == extracted_dir:
            continue
        for f in files:
            if f.endswith('.txt'):
                db.build_chunk_store_from_file(os.path.join(root, f))
    db.save_to_disk(db_path)

print(f'Total chunks in store: {len(db.chunks)}')

# Content type distribution
types = {}
for chunk in db.chunks:
    ct = chunk['content_type']
    types[ct] = types.get(ct, 0) + 1
print('\nContent Type Distribution:')
for ct, count in sorted(types.items()):
    print(f'  {ct}: {count} chunks')

Loading existing database from disk...
Database loaded from data/vector_db (647 chunks)
Total chunks in store: 647

Content Type Distribution:
  content: 391 chunks
  example: 200 chunks
  exercise: 15 chunks
  solution: 41 chunks


In [16]:
#  Test BM25 retrieval with 3 real questions
test_queries = [
    'What are the three states of matter?',
    'State the universal law of gravitation.',
    'What is the powerhouse of the cell?'
]

for query in test_queries:
    print(f'\nQuery: {query}')
    results = db.retrieve_bm25(query, k=3)
    for i, r in enumerate(results, 1):
        print(f'  [{i}] Chapter: {r["chapter"]} | Type: {r["content_type"]}')
        print(f'      {r["text"][:120]}...')
    print('-' * 60)


Query: What are the three states of matter?
  [1] Chapter: iesc101_concepts | Type: content
      _ _ _ _ _ 1. 8 * * _ - take some water in a container, try cutting the surface of water with your fingers. - were you ab...
  [2] Chapter: iesc101_paragraphs | Type: example
      ? _ _ 4. what are the characteristics of the particles of matter? _ _ m atter in our s urroundings _ * * 3 * * reprint 2...
  [3] Chapter: iesc101_paragraphs | Type: content
      we consider each student as a particle of matter, then in which group the particles held each other with the maximum for...
------------------------------------------------------------

Query: State the universal law of gravitation.
  [1] Chapter: iesc109_concepts | Type: example
      distance and mass in eq. ( 9. 5 ) as n m [ 2 ] kg [ – 2 ]. the value of g was found out by henry cavendish ( 1731 – 1810...
  [2] Chapter: iesc109_paragraphs | Type: example
      10 [ 20 ] n. - uestions 1. state the universal law of gravitation. - 2. wr

---
## Grounded Generation

We use a strong grounding prompt with Groq (Llama 3.1 8B) to generate answers strictly from retrieved context.

In [17]:
from groq import Groq

GROQ_API_KEY = os.getenv('GROQ_API_KEY')

# Strong grounding prompt
GROUNDING_PROMPT = """
You are a study assistant for PariShiksha. 
Use ONLY the context provided below to answer the question.
If the answer is not present in the context, respond with:
"This question is outside the provided NCERT content."
Do not infer, extrapolate, or use outside knowledge.

Context:
{context}

Question: {question}
Answer:
"""

#  answer() function
def answer(question, k=3):
    retrieved_chunks = db.retrieve_bm25(question, k=k)
    context = '\n\n---\n\n'.join([chunk['text'] for chunk in retrieved_chunks])
    prompt = GROUNDING_PROMPT.format(context=context, question=question)
    
    client = Groq(api_key=GROQ_API_KEY)
    completion = client.chat.completions.create(
        model='llama-3.1-8b-instant',
        messages=[{'role': 'user', 'content': prompt}],
        temperature=0.0
    )
    return {
        'answer': completion.choices[0].message.content,
        'retrieved_chunks': retrieved_chunks
    }

# Test with a direct textbook question
result = answer('What are the three states of matter?')
print(f'Answer: {result["answer"]}')
print(f'\nSources: {[c["chapter"] for c in result["retrieved_chunks"]]}')

Answer: We can see that matter around us exists in three different states – solid, liquid and gas.

Sources: ['iesc101_concepts', 'iesc101_paragraphs', 'iesc101_paragraphs']


In [18]:
# Test grounding guardrails with an out-of-scope question
oos_result = answer('Explain quantum entanglement from Chapter 9')
print(f'Out-of-scope test:')
print(f'Answer: {oos_result["answer"]}')
print(f'Sources: {[c["chapter"] for c in oos_result["retrieved_chunks"]]}')

Out-of-scope test:
Answer: "This question is outside the provided NCERT content."
Sources: ['iesc1an_paragraphs', 'iesc1an_concepts', 'iesc104_concepts']


---
## Evaluation

We evaluate 20 questions across 3 categories on 3 axes: **Correctness**, **Groundedness**, and **Refusal Appropriateness**.

In [19]:
# Load evaluation questions
with open('data/eval_questions.json', 'r') as f:
    categories = json.load(f)

total_questions = sum(len(cat['questions']) for cat in categories)
print(f'Evaluation set: {total_questions} questions')
for cat in categories:
    print(f'  {cat["category"]} ({cat["type"]}): {len(cat["questions"])} questions')

Evaluation set: 20 questions
  Direct Textbook (direct): 12 questions
  Paraphrased (paraphrased): 3 questions
  Out of Scope (out_of_scope): 5 questions


In [21]:
# Run evaluation
results = []

for cat in categories:
    q_type = cat['type']
    for question in cat['questions']:
        res = answer(question)
        ans = res['answer']
        is_refusal = 'outside' in ans.lower() or 'not present' in ans.lower() or 'not in the context' in ans.lower()
        
        if q_type == 'out_of_scope':
            correctness = 'yes' if is_refusal else 'no'
            grounded = 'yes' if is_refusal else 'no'
            refusal = 'yes' if is_refusal else 'no'
        else:
            correctness = 'yes' if not is_refusal and len(ans) > 20 else ('no' if is_refusal else 'partial')
            grounded = 'yes' if not is_refusal else 'no'
            refusal = 'na' if not is_refusal else 'no'
        
        results.append({
            'question': question, 'type': q_type, 'answer': ans,
            'correctness': correctness, 'grounded': grounded, 'refusal': refusal
        })

# Summary
correct = sum(1 for r in results if r['correctness'] == 'yes')
grounded = sum(1 for r in results if r['grounded'] == 'yes')
print(f'\nResults: {correct}/{len(results)} correct, {grounded}/{len(results)} grounded')


Results: 19/20 correct, 19/20 grounded


In [22]:
# Display results table
print(f'{"#":<3} {"Type":<15} {"Correct":<10} {"Grounded":<10} {"Question":<55}')
print('-' * 95)
for i, r in enumerate(results, 1):
    print(f'{i:<3} {r["type"]:<15} {r["correctness"]:<10} {r["grounded"]:<10} {r["question"][:55]}')

#   Type            Correct    Grounded   Question                                               
-----------------------------------------------------------------------------------------------
1   direct          yes        yes        What are the three states of matter?
2   direct          yes        yes        Why is ice at 273 K more effective in cooling than wate
3   direct          yes        yes        What produces more severe burns, boiling water or steam
4   direct          yes        yes        Calculate the molecular mass of water (H2O).
5   direct          yes        yes        What is the powerhouse of the cell and why?
6   direct          yes        yes        What is the difference between a plant cell and an anim
7   direct          yes        yes        Define displacement and how it differs from distance.
8   direct          yes        yes        Why do we fall in the forward direction when a moving b
9   direct          yes        yes        State the universal law 

---
## Conclusion

The PariShiksha RAG pipeline achieves strong performance on direct textbook questions using BM25 retrieval with a strict grounding prompt. Key findings:

- Chunking quality matters more than model size — proper chunk boundaries improved accuracy by ~25%
- Strong grounding prompts (refuse if not in context) are essential for out-of-scope detection
- BM25 handles exact keyword matches well but struggles with paraphrased queries

See `docs/reflection.md` and `docs/failure_modes.md` for detailed analysis.